In [1]:
from _setup import setup_project_root
PROJECT_ROOT = setup_project_root()

In [2]:
import importlib
import pandas as pd
import joblib
import evaluation.metrics as metrics
from models.XGBoost import XGBoostBaseConfig, build_xy, fit_model, predict

In [3]:
train_path = PROJECT_ROOT / "data" / "station_split" / "train.csv"
val_path   = PROJECT_ROOT / "data" / "station_split" / "val.csv"
test_path  = PROJECT_ROOT / "data" / "station_split" / "test.csv"

train_df = pd.read_csv(train_path)
val_df   = pd.read_csv(val_path)
test_df  = pd.read_csv(test_path)

In [4]:
extra_features = [
    # Cảm biến gốc
    'TSP', 'O3', 'CO', 'NO2', 'SO2', 'Temperature', 'Humidity', 
    
    # Thời gian
    'hour', 'day_of_week', 'hour_sin', 'hour_cos', 
    
    # PM2.5 Multi-lags (Bắt quán tính)
    'PM2.5_lag1', 'PM2.5_lag2', 'PM2.5_lag3', 'PM2.5_lag6', 'PM2.5_lag12', 'PM2.5_lag24',
    
    # PM2.5 Rolling Stats (Bắt xu hướng và biến động)
    'PM2.5_roll3_mean', 'PM2.5_roll6_mean', 'PM2.5_roll12_mean',
    'PM2.5_roll3_std', 'PM2.5_roll6_std', 'PM2.5_roll12_std',
    
    # Khí tượng Lags (Yếu tố quyết định Correlation 0.7)
    'Temperature_lag1', 'Temperature_lag3', 
    'Humidity_lag1', 'Humidity_lag3',
    'NO2_lag1', 'NO2_lag3',
    'SO2_lag1', 'SO2_lag3'
]

In [5]:
pm25_scaler = joblib.load(PROJECT_ROOT / "artifacts" / "pm25_scaler.pkl")

In [6]:
config = XGBoostBaseConfig(time_col="date", lags=24, horizon=24)

In [7]:
X_train, y_train = build_xy(train_df, config, extra_feature_cols=extra_features)
X_val, y_val     = build_xy(val_df, config, extra_feature_cols=extra_features)
X_test, y_test   = build_xy(test_df, config, extra_feature_cols=extra_features)

In [8]:
model = fit_model(X_train, y_train, config)

In [9]:
from evaluation.metrics import compute_metrics, compute_metrics_real_scale

In [10]:
preds = predict(model, X_test)
metrics_res = compute_metrics_real_scale(y_test, preds, scaler=pm25_scaler)

In [37]:
metrics_station_split = compute_metrics_real_scale(
    y_test, pred_test, scaler=pm25_scaler, mape_threshold=1.0
)
metrics_station_split

{'Correlation': 0.550625698201664,
 'RMSE': 0.14628072171130255,
 'MAPE': 0.4045777702970164,
 'MAE': 0.10576026370505373}

In [38]:
metrics_station_split_1step = compute_metrics_real_scale(
    y_test[:,0], pred_test[:,0], scaler=pm25_scaler, mape_threshold=1.0
)
metrics_station_split_1step


{'Correlation': 0.8342745598961717,
 'RMSE': 0.09561221609255098,
 'MAPE': 0.14779159937586103,
 'MAE': 0.056794465397963456}

In [30]:
# Thay 'model' bằng tên mô hình bạn đã train (ví dụ: regressor, model_gru, ...)
# Thay 'X_test' bằng tập đặc trưng dùng để kiểm thử của bạn
pred_test = model.predict(X_test)

# Kiểm tra thử kích thước sau khi dự báo
print(f"Kích thước dự báo: {pred_test.shape}")

Kích thước dự báo: (6813, 24)


In [36]:
import numpy as np
import pandas as pd

if 'pred_test' not in locals() or 'y_test' not in locals():
    print("❌ Bạn cần chạy bước dự báo (model.predict) trước!")
else:
    station_ids = test_df["Station_No"].values[-len(y_test):]
    rows = []
    for sid in np.unique(station_ids):
        mask = (station_ids == sid)
        
        if mask.sum() < 50: 
            continue
            
        y_true_sub = y_test[mask, 0] if y_test.ndim > 1 else y_test[mask]
        y_pred_sub = pred_test[mask, 0] if pred_test.ndim > 1 else pred_test[mask]
        
        try:
            met = compute_metrics_real_scale(
                y_true_sub, 
                y_pred_sub, 
                scaler=pm25_scaler, 
                mape_threshold=1.0
            )
            rows.append({"Station_No": sid, "n": int(mask.sum()), **met})
        except Exception as e:
            continue

    if rows:
        df_station_metrics = pd.DataFrame(rows).sort_values("RMSE").reset_index(drop=True)
        display(df_station_metrics.head(10)) 
    else:
        print("❌ Không tìm thấy dữ liệu phù hợp.")

,Station_No,n,Correlation,RMSE,MAPE,MAE
0,5,1135,0.838991,0.092864,0.112415,0.055565
1,4,1136,0.834912,0.093699,0.278461,0.055774
2,2,1135,0.849775,0.095421,0.143080,0.057287
3,6,1135,0.830437,0.096356,0.092291,0.057079
4,1,1136,0.826619,0.096885,0.124226,0.056685
5,3,1136,0.823725,0.098338,0.142722,0.058375
